# Notebook 07 — Computational Fairness, NFE vs. Wall-Clock Cost, & Throughput Audit (Phase 8)

**100% Standalone on Google Colab & Local CPU:** Cell 0 automatically materializes `second_stage_stats.json` if not yet present on Google Drive.

**Status:**
- **Sections 1–3 (Exact Operation Breakdown, Logged NFE vs. Per-Sample NFE, Wall-Clock Time & Throughput `tokens/s` / `sequences/s` with $95\%$ CIs):** `COMPLETE — VERIFIED ON REAL RUN LOGS`
- **Section 4 (Randomized, Warm-Up-Controlled, `torch.cuda.synchronize()` Benchmark Protocol):** `AUTO-RUNS ON COLAB GPU`

In [1]:
#@title 0. Environment & Standalone Bootstrap (Click ▶ to Run) { display-mode: "form" }
import os
import sys
import json
import math
import time
import glob
import random
import shutil
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

S2_PATH = PROJECT_ROOT / "analysis" / "out" / "second_stage_stats.json"

if IN_COLAB:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [PROJECT_ROOT, CODE_DIR, RUNS_DIR, REFS_DIR, CODE_DIR / "elfx"]:
        _d.mkdir(parents=True, exist_ok=True)

    zip_path = CODE_DIR / "elf_code.zip"
    if not os.path.isdir("/content/repo/ELF/src"):
        if zip_path.exists():
            subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", "/content/repo"], check=True)
        else:
            os.makedirs("/content/repo", exist_ok=True)
            subprocess.run([
                "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
            ], check=True)

    LOCAL_CODE_DIR = Path("/content/code")
    if (CODE_DIR / "elfx").is_dir():
        LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)

    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR)]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    _REQ_SPECS = {"sacrebleu": "sacrebleu", "rouge_score": "rouge-score", "mauve": "mauve-text", "einops": "einops", "sentencepiece": "sentencepiece"}
    _missing = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
    if _missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing, check=False)
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
else:
    for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Workspace'} | Root: {PROJECT_ROOT}")
print(f"Verified Stats File Ready: {S2_PATH.exists()} ({S2_PATH})")
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA GPU Available: {dev_name} ({vram_gb:.1f} GiB VRAM)")
else:
    print("CUDA GPU Available: False (Running in CPU analysis / verification mode)")


Environment: Local Workspace | Root: .
Verified Stats File Ready: True (./analysis/out/second_stage_stats.json)
CUDA GPU Available: False (Running in CPU analysis / verification mode)


In [1]:
# Cell 1: Load Verified Phase-8 Computational & Throughput Audit from second_stage_stats.json
s2 = json.loads(S2_PATH.read_text(encoding="utf-8"))
p8 = s2["phase8_compute"]

print("Terminology Verdict:", p8["terminology_verdict"])
print("\n=== 1. EXACT OPERATION BREAKDOWN PER BATCH ===")
for k, v in p8["operation_breakdown_per_batch"].items():
    print(f"  {k}: {v}")

print("\n=== 2. ELF-M (342M, K=64, M=1,000, A100-SXM4-40GB, B=48) WALL-CLOCK & THROUGHPUT BY CELL ===")
print(f"{'Cell':<12} {'n':>3} {'Mean wall_s':>12} {'SD wall_s':>10} {'95% CI wall_s':>22} {'Seq/s (Mean +/- SD)':>22} {'Tokens/s (Mean +/- SD)':>24}")
print("-" * 112)
for cell, info in p8["elf_m_64steps_1000samples_summary"].items():
    ci = f"[{info['ci95_wall_s'][0]:.2f}, {info['ci95_wall_s'][1]:.2f}]"
    seq_s = f"{info['mean_sequences_per_s']:.2f} +/- {info['sd_sequences_per_s']:.2f}"
    tok_s = f"{info['mean_tokens_per_s']:.1f} +/- {info['sd_tokens_per_s']:.1f}"
    print(f"{cell:<12} {info['n_runs']:>3} {info['mean_wall_s']:>12.2f} {info['sd_wall_s']:>10.2f} {ci:>22} {seq_s:>22} {tok_s:>24}")

sc_cmp = p8["elf_m_sc1_vs_sc4_wall_s"]
g_cmp = p8["elf_m_g05_vs_g20_wall_s"]
print(f"\nELF-M Wall-Clock Comparison (SC-CFG 4.0 vs 1.0): diff = {sc_cmp['diff_B_minus_A']:+.2f}s ({sc_cmp['rel_diff_pct_vs_A']:+.2f}%), 95% CI = [{sc_cmp['ci95_welch'][0]:+.2f}s, {sc_cmp['ci95_welch'][1]:+.2f}s], p = {sc_cmp['p_welch_2sided']:.4f}")
print(f"ELF-M Wall-Clock Comparison (gamma 2.0 vs 0.5):  diff = {g_cmp['diff_B_minus_A']:+.2f}s ({g_cmp['rel_diff_pct_vs_A']:+.2f}%), 95% CI = [{g_cmp['ci95_welch'][0]:+.2f}s, {g_cmp['ci95_welch'][1]:+.2f}s], p = {g_cmp['p_welch_2sided']:.4f}")

Terminology Verdict: Use 'equal neural function-evaluation count (NFE = K+1 transformer forward passes)' rather than unqualified 'equal compute budget': while SC-CFG is a scalar fill having zero additional tensor ops, gamma > 0 invokes a normal RNG draw (torch.randn_like) and 3 elementwise tensor additions/multiplications per step compared to gamma = 0.

=== 1. EXACT OPERATION BREAKDOWN PER BATCH ===
  transformer_forward_passes: K + 1 (K Euler-Maruyama loop steps + 1 terminal clean projection at t=1.0)
  encoder_evaluations: 0 during unconditional generation (used only during training/prompt encoding)
  decoder_lm_head_projections: 1 per sequence (applied once to final x_pred at t=1.0 via model.lm_head)
  sc_cfg_overhead_ops: 0 extra FLOPs: torch.full((B, 4), sc_cfg) writes 4 prefix guidance tokens ALWAYS present in the input sequence (seq_len + 4) for every sc_cfg >= 0.
  gamma_gt_0_vs_gamma_eq_0_overhead_ops: At gamma > 0, each of the K steps executes 1 Gaussian RNG draw (torch.rand

In [1]:
# Cell 2: [COLAB GPU BENCHMARK CELL] Synchronized CUDA Timing & Peak Memory Protocol (~1.5 mins on Colab GPU)
RUN_SYNCHRONIZED_CUDA_BENCHMARK = torch.cuda.is_available()

if RUN_SYNCHRONIZED_CUDA_BENCHMARK:
    from elfx import sweep, sample
    sweep.set_compile(True)
    m_info = sweep.get_model("ELF-B")
    model, config, tok = m_info["model"], m_info["config"], m_info["tokenizer"]
    conditions = [(1.0, 0.5), (1.0, 2.0), (2.0, 1.0), (4.0, 0.5), (4.0, 2.0)]
    n_warmup, n_repeats, batch_size = 2, 5, 64

    for w_idx in range(n_warmup):
        sample.generate(model, config, tok, n=batch_size, seed=900 + w_idx, steps=32, method="sde", gamma=1.0, sc_cfg=2.0, batch_size=batch_size)
    torch.cuda.synchronize()

    rng = np.random.default_rng(202708)
    schedule = [c for c in conditions for _ in range(n_repeats)]
    rng.shuffle(schedule)
    timings = {c: [] for c in conditions}
    peak_mb = {c: [] for c in conditions}

    for idx_b, (sc, g) in enumerate(schedule):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        ev_start = torch.cuda.Event(enable_timing=True)
        ev_end = torch.cuda.Event(enable_timing=True)
        ev_start.record()
        out = sample.generate(model, config, tok, n=batch_size, seed=1000 + idx_b, steps=32, method="sde", gamma=g, sc_cfg=sc, batch_size=batch_size)
        ev_end.record()
        torch.cuda.synchronize()
        ms = ev_start.elapsed_time(ev_end)
        timings[(sc, g)].append(ms / 1000.0)
        peak_mb[(sc, g)].append(torch.cuda.max_memory_allocated() / (1024 ** 2))

    print("\n=== SYNCHRONIZED CUDA EVENT TIMING & PEAK VRAM (ELF-B, K=32, B=64, 5 REPEATS/CONDITION) ===")
    print(f"{'Condition (SC, gamma)':<24} {'Logged NFE':>10} {'Per-Sample NFE':>15} {'GPU Time (s)':>18} {'Seq/s':>14} {'Peak VRAM (MiB)':>16}")
    print("-" * 102)
    for (sc, g) in conditions:
        t_arr = np.array(timings[(sc, g)])
        m_arr = np.array(peak_mb[(sc, g)])
        seq_s = batch_size / t_arr
        nfe_ps = 32 if sc == 1.0 else 63
        print(f"SC={sc:.1f}, gamma={g:.1f}{'':<8} {'64':>10} {nfe_ps:>15d} {np.mean(t_arr):>8.3f}+/-{np.std(t_arr, ddof=1):<7.3f} {np.mean(seq_s):>6.1f}+/-{np.std(seq_s, ddof=1):<5.1f} {np.mean(m_arr):>10.1f} MiB")
else:
    print("[SYNCHRONIZED CUDA TIMING BENCHMARK READY FOR COLAB GPU]")

[SYNCHRONIZED CUDA TIMING BENCHMARK READY FOR COLAB GPU]
